In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path

class SUIMDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        self.root_dir = Path(root_dir)
        self.split = split
        self.transform = transform
        self.images_dir = self.root_dir / split / 'images'
        self.masks_dir = self.root_dir / split / 'masks'
        self.image_files = sorted([f for f in os.listdir(self.images_dir) if f.endswith(('.jpg', '.png', '.bmp'))])
        print(f"Found {len(self.image_files)} images in {split} set")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = self.images_dir / img_name
        image = Image.open(img_path).convert('RGB')

        mask_name = img_name.replace('.jpg', '.bmp').replace('.png', '.bmp')
        mask_path = self.masks_dir / mask_name
        if not mask_path.exists():
            mask_name = img_name.replace('.jpg', '.png')
            mask_path = self.masks_dir / mask_name
        mask = Image.open(mask_path).convert('L')

        if self.transform:
            image = self.transform(image)
            mask = transforms.ToTensor()(mask)
            mask = (mask * 255).long().squeeze(0)
            mask = remap_mask(mask)
        else:
            image = transforms.ToTensor()(image)
            mask = transforms.ToTensor()(mask)
            mask = (mask * 255).long().squeeze(0)
            mask = remap_mask(mask)

        return image, mask

print("SUIMDataset class defined successfully!")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path

class SUIMDataset(Dataset):
    def __init__(self, root_dir, image_files, transform=None):
        self.root_dir = Path(root_dir)
        self.image_files = image_files
        self.transform = transform

        self.images_dir = self.root_dir / 'images'
        self.masks_dir = self.root_dir / 'masks'

        print(f"Loaded {len(self.image_files)} images")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = self.images_dir / img_name
        image = Image.open(img_path).convert('RGB')

        # Get corresponding mask (change extension to .png)
        mask_name = img_name.replace('.jpg', '.png').replace('.bmp', '.png')
        mask_path = self.masks_dir / mask_name
        mask = Image.open(mask_path).convert('L')

        # Resize BOTH image and mask to same size
        image = image.resize((256, 256), Image.BILINEAR)
        mask = mask.resize((256, 256), Image.NEAREST)

        # Apply transforms to image
        if self.transform:
            image = self.transform(image)
        else:
            image = transforms.ToTensor()(image)

        # Convert mask to tensor
        mask = transforms.ToTensor()(mask)
        mask = (mask * 255).long().squeeze(0)
        mask = remap_mask(mask)

        return image, mask


# Define transforms (WITHOUT Resize since we do it manually)
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Get dataset path
import kagglehub
base_path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")
data_path = os.path.join(base_path, 'dataset')

print(f"Using data path: {data_path}\n")

# Get all image files
images_dir = os.path.join(data_path, 'images')
all_image_files = sorted([f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.png', '.bmp'))])

print(f"Total images found: {len(all_image_files)}")

# Split into train and test (80-20 split)
num_train = int(0.8 * len(all_image_files))
train_files = all_image_files[:num_train]
test_files = all_image_files[num_train:]

print(f"Train images: {len(train_files)}")
print(f"Test images: {len(test_files)}\n")

# Create datasets
train_dataset = SUIMDataset(root_dir=data_path, image_files=train_files, transform=train_transform)
test_dataset = SUIMDataset(root_dir=data_path, image_files=test_files, transform=test_transform)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=0)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of testing batches: {len(test_loader)}\n")


# Class names and colors
class_names = [
    'Background',
    'Human divers',
    'Aquatic plants',
    'Wrecks/ruins',
    'Robots',
    'Reefs/invertebrates',
    'Fish/vertebrates',
    'Sea-floor/rocks'
]

colors = [
    [0, 0, 0],
    [255, 0, 0],
    [0, 255, 0],
    [0, 0, 255],
    [255, 255, 0],
    [255, 0, 255],
    [0, 255, 255],
    [128, 128, 128]
]

# Helper functions
def denormalize(img):
    img = img.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = std * img + mean
    return np.clip(img, 0, 1)

def mask_to_rgb(mask):
    h, w = mask.shape
    rgb_mask = np.zeros((h, w, 3), dtype=np.uint8)
    for class_id, color in enumerate(colors):
        rgb_mask[mask == class_id] = color
    return rgb_mask

# Get a batch
dataiter = iter(train_loader)
images, masks = next(dataiter)

# Display samples
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for i in range(4):
    # Original image
    ax = axes[0, i]
    img = denormalize(images[i])
    ax.imshow(img)
    ax.set_title(f'Image {i+1}', fontsize=12)
    ax.axis('off')

    # Mask
    ax = axes[1, i]
    mask = masks[i].numpy()
    rgb_mask = mask_to_rgb(mask)
    ax.imshow(rgb_mask)
    ax.set_title(f'Ground Truth Mask', fontsize=12)
    ax.axis('off')

    # Overlay
    ax = axes[2, i]
    ax.imshow(img)
    ax.imshow(rgb_mask, alpha=0.5)
    ax.set_title(f'Overlay', fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.show()

# Print class distribution in a sample mask
print(f"\nClass distribution in first mask:")
unique, counts = torch.unique(masks[0], return_counts=True)
for cls, count in zip(unique.numpy(), counts.numpy()):
    if cls < len(class_names):
        print(f"  Class {cls} ({class_names[cls]}): {count} pixels")

In [ ]:
# TO DO
# Install segmentation_models_pytorch if not already installed
!pip install segmentation-models-pytorch -q

import segmentation_models_pytorch as smp

# Define the model
class UNetSegmentation(nn.Module):
    def __init__(self, num_classes=8, encoder_name='efficientnet-b1', encoder_weights='imagenet'):
        super(UNetSegmentation, self).__init__()

        self.model = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=num_classes,
            activation=None  # We'll apply softmax in the loss function
        )

    def forward(self, x):
        return self.model(x)


# Create model instance
num_classes = 8
model = UNetSegmentation(num_classes=num_classes)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model created successfully!")
print(f"Encoder: efficientnet-b1")
print(f"Number of classes: {num_classes}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# TO DO
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    """
    Train the model for one epoch.

    Returns:
        avg_loss: Average training loss
        avg_iou: Average IoU score
    """
    model.train()
    running_loss = 0.0
    running_iou = 0.0
    num_batches = 0

    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, masks)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Calculate IoU
        with torch.no_grad():
            pred_masks = torch.argmax(outputs, dim=1)
            iou = calculate_iou(pred_masks, masks, num_classes)

        # Statistics
        running_loss += loss.item()
        running_iou += iou
        num_batches += 1

    avg_loss = running_loss / num_batches
    avg_iou = running_iou / num_batches

    return avg_loss, avg_iou


def validate(model, test_loader, criterion, device):
    """
    Validate the model.

    Returns:
        avg_loss: Average validation loss
        avg_iou: Average IoU score
    """
    model.eval()
    running_loss = 0.0
    running_iou = 0.0
    num_batches = 0

    with torch.no_grad():
        for images, masks in test_loader:
            images = images.to(device)
            masks = masks.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, masks)

            # Calculate IoU
            pred_masks = torch.argmax(outputs, dim=1)
            iou = calculate_iou(pred_masks, masks, num_classes)

            # Statistics
            running_loss += loss.item()
            running_iou += iou
            num_batches += 1

    avg_loss = running_loss / num_batches
    avg_iou = running_iou / num_batches

    return avg_loss, avg_iou


def calculate_iou(pred, target, num_classes):
    """
    Calculate mean Intersection over Union (IoU) for all classes.
    """
    ious = []
    pred = pred.cpu().numpy()
    target = target.cpu().numpy()

    for cls in range(num_classes):
        pred_cls = (pred == cls)
        target_cls = (target == cls)

        intersection = np.logical_and(pred_cls, target_cls).sum()
        union = np.logical_or(pred_cls, target_cls).sum()

        if union == 0:
            iou = 1.0  # If no ground truth and no prediction, perfect score
        else:
            iou = intersection / union

        ious.append(iou)

    return np.mean(ious)


print("Training and validation functions defined successfully!")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

# Move model to device
model = model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# Training parameters
num_epochs = 20

# Lists to store metrics
train_losses = []
val_losses = []
train_ious = []
val_ious = []

# Training loop
print("Starting training...\n")
best_val_iou = 0.0

for epoch in range(num_epochs):
    # Train
    train_loss, train_iou = train_one_epoch(model, train_loader, criterion, optimizer, device)

    # Validate
    val_loss, val_iou = validate(model, test_loader, criterion, device)

    # Step scheduler
    old_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_loss)
    new_lr = optimizer.param_groups[0]['lr']

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_ious.append(train_iou)
    val_ious.append(val_iou)

    # Save best model
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save(model.state_dict(), 'best_unet_suim.pth')

    # Print progress
    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"  Train Loss: {train_loss:.4f} | Train IoU: {train_iou:.4f}")
    print(f"  Val Loss:   {val_loss:.4f} | Val IoU:   {val_iou:.4f}")
    if new_lr < old_lr:
        print(f"  Learning rate reduced: {old_lr:.6f} -> {new_lr:.6f}")
    print()

print("Training completed!")
print(f"Best Validation IoU: {best_val_iou:.4f}")

# Plot metrics
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
epochs_range = range(1, num_epochs + 1)
ax1.plot(epochs_range, train_losses, 'b-o', label='Training Loss', linewidth=2)
ax1.plot(epochs_range, val_losses, 'r-o', label='Validation Loss', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# IoU plot
ax2.plot(epochs_range, train_ious, 'b-o', label='Training IoU', linewidth=2)
ax2.plot(epochs_range, val_ious, 'r-o', label='Validation IoU', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('IoU', fontsize=12)
ax2.set_title('Training and Validation IoU', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal Results:")
print(f"Training Loss: {train_losses[-1]:.4f}")
print(f"Training IoU: {train_ious[-1]:.4f}")
print(f"Validation Loss: {val_losses[-1]:.4f}")
print(f"Validation IoU: {val_ious[-1]:.4f}")

In [ ]:
# TO DO
from matplotlib.patches import Patch

# Load best model
model.load_state_dict(torch.load('best_unet_suim.pth'))
model.eval()

# Get a batch from test set
dataiter = iter(test_loader)
images, masks = next(dataiter)

# Move to device and get predictions
images_gpu = images.to(device)
with torch.no_grad():
    outputs = model(images_gpu)
    pred_masks = torch.argmax(outputs, dim=1)

# Move back to CPU
images = images.cpu()
masks = masks.cpu()
pred_masks = pred_masks.cpu()

# Visualize predictions
num_samples = min(6, len(images))
fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))

# Handle single sample case
if num_samples == 1:
    axes = axes.reshape(1, -1)

for i in range(num_samples):
    # Denormalize image
    img = denormalize(images[i])

    # Get masks
    gt_mask = masks[i].numpy()
    pred_mask = pred_masks[i].numpy()

    # Convert to RGB
    rgb_gt = mask_to_rgb(gt_mask)
    rgb_pred = mask_to_rgb(pred_mask)

    # Calculate IoU for this sample
    iou = calculate_iou(pred_masks[i:i+1], masks[i:i+1], num_classes)

    # Plot
    axes[i, 0].imshow(img)
    axes[i, 0].set_title('Original Image', fontsize=12)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(rgb_gt)
    axes[i, 1].set_title('Ground Truth', fontsize=12)
    axes[i, 1].axis('off')

    axes[i, 2].imshow(rgb_pred)
    axes[i, 2].set_title(f'Prediction (IoU: {iou:.3f})', fontsize=12)
    axes[i, 2].axis('off')

    axes[i, 3].imshow(img)
    axes[i, 3].imshow(rgb_pred, alpha=0.5)
    axes[i, 3].set_title('Overlay', fontsize=12)
    axes[i, 3].axis('off')

plt.tight_layout()
plt.show()

# Display class legend
fig, ax = plt.subplots(1, 1, figsize=(12, 3))
ax.axis('off')

# Create legend patches
legend_elements = [
    Patch(facecolor=np.array(color)/255, label=f'{i}: {name}')
    for i, (name, color) in enumerate(zip(class_names, colors))
]

ax.legend(handles=legend_elements, loc='center', ncol=4, fontsize=11)
ax.set_title('Class Legend', fontsize=14, fontweight='bold')
plt.show()

# Print overall metrics
print(f"\nOverall Test Set Performance:")
print(f"Best Validation IoU: {best_val_iou:.4f}")
print(f"Final Validation IoU: {val_ious[-1]:.4f}")